# Helmholtz eigenvalue problem in a rectangle

$$
\mathbb{S}_u
\begin{cases}
\Omega = [0, L_x] \times [0, L_y] \\
u_{\text{D}}\vert_{\partial\Omega}=0 \\
f = 0 \\
\end{cases}
$$

In [ ]:
import numpy as np
from dolfinx.fem import FunctionSpace
from lucifex.mesh import ellipse_mesh, mesh_boundary
from lucifex.solver import (
    evp, BoundaryConditions, OptionsSLEPc,
)
from lucifex.plt import plot_colormap, save_figure
from lucifex.pde.eigen import helmholtz

r = 1.0
Nr = 32
h = r / Nr
mesh = ellipse_mesh(h)(r)
boundary = mesh_boundary(
        mesh, 
        {
            "outer": lambda x: x[0]**2 + x[1] **2 - r**2,
        },
)
bcs = BoundaryConditions(('dirichlet', boundary.union, 0.0))
fs = FunctionSpace(mesh, ('P', 1))

nev = 10
slepc = OptionsSLEPc(
    eps_tol=1e-10,
    eps_target=0.0,
    eps_nev=nev,
    eps_ncv=50,
    eps_max_it=1000,
    eps_which='smallest_real',
)
u_solver = evp(helmholtz, bcs, slepc)(fs)
u_solver.solve()

In [ ]:
sqrt = lambda x: np.sqrt(np.real(x)) if np.real(x) > 0 else None
k_values = [sqrt(-e) for e in u_solver.eigenvalues]
k_values

In [ ]:
for k, u in zip(k_values, u_solver.eigenfunctions):
    if k is not None:
        fig, ax = plot_colormap(u, title=f'$u_{{k={k:.3f}}}$')
        save_figure(f'u(x,y)_k={k:.3f}', thumbnail=(k is k_values[-1]))(fig)